# 22 · Rerank · HyDE · Multi-query

> **学习目标**：实现 RAG 高级阶段的 3 个常见技巧：(a) 用 LLM-as-judge 当 reranker（避开 bge-reranker 装不上的坑）；(b) HyDE 让 LLM 先写「假答案」再检索；(c) multi-query 把一个 query 改写成 N 个再合并。
>
> **预备**：21（hybrid search）跑过。OFFLINE 用 stub，ONLINE 切到 Ollama。
>
> **为什么重要**：这三招是 RAG 生产环境从「能用」走向「好用」的标配。每一招都比换 embedding / 调 chunk 收益更大、风险更小。

In [ ]:
MODE = 'OFFLINE'        # 'ONLINE' = 用 Ollama LLM 做 reranker / HyDE / multi-query

import numpy as np, hashlib, re, requests, json
OLLAMA = 'http://127.0.0.1:11434'

def fake_embed(text, dim=256):
    seed = int(hashlib.sha256(text.encode('utf-8')).hexdigest()[:8], 16)
    v = np.random.default_rng(seed).standard_normal(dim).astype(np.float32)
    return v / (np.linalg.norm(v) + 1e-12)

def ollama_embed(text):
    r = requests.post(f'{OLLAMA}/api/embeddings', json={'model':'nomic-embed-text','prompt':text}, timeout=30)
    r.raise_for_status()
    v = np.array(r.json()['embedding'], dtype=np.float32)
    return v / (np.linalg.norm(v) + 1e-12)

def ollama_chat(prompt, model='qwen1.5_1.8'):
    r = requests.post(f'{OLLAMA}/api/chat', json={
        'model': model, 'stream': False, 'options': {'temperature': 0.0},
        'messages': [{'role':'user','content':prompt}]
    }, timeout=120)
    r.raise_for_status()
    return r.json()['message']['content']

if MODE == 'ONLINE':
    try:
        requests.get(f'{OLLAMA}/api/tags', timeout=1).raise_for_status()
        embed = ollama_embed
        chat  = ollama_chat
        print('✅ ONLINE：Ollama 提供 embed + LLM')
    except Exception:
        MODE = 'OFFLINE'
        print('⚠ Ollama 未启动，降级 OFFLINE')
if MODE == 'OFFLINE':
    embed = fake_embed
    # 简陋 LLM stub：用规则模拟 reranker / HyDE / multi-query 的输出
    def chat(prompt):
        return f'[STUB] {prompt[:80]}'
    print('使用 OFFLINE 模式（LLM 用规则模拟，结果只为验证流程）')

In [ ]:
# 沿用 21 的语料 + dense baseline
DOCS = [
    '锂电池 XZ4054H 是一款高能量密度电池，容量 300mAh，循环寿命 500 次。',
    'XZ5352R 是一款 800mAh 锂电池，专为高放电场景设计。',
    'XZ4054H-NE1.11 是 XZ4054H 的升级版，集成保护电路，规格书 V1.11。',
    '电源管理芯片 PM8916 提供多路稳压输出，工作电压 3.3V。',
    'Transformer 是 2017 年由 Google 提出的注意力机制神经网络。',
    'RAG 是检索增强生成的简称，把外部知识接进 LLM 上下文。',
    '注意力机制（Attention）解决了 RNN 在长序列上的梯度衰减问题。',
    'PM8917 是 PM8916 的低功耗替代型号，待机电流降低 50%。',
]
doc_vecs = np.vstack([embed(d) for d in DOCS])

def dense_search(query, top_k=5):
    q = embed(query)
    sims = doc_vecs @ q
    return [(int(i), float(sims[i])) for i in np.argsort(-sims)[:top_k]]

## 1. Rerank —— 用 LLM-as-judge 做精排

**思路**：
1. 粗召回 top-20（用 dense / hybrid）
2. 对每个候选 (query, doc) 让 LLM 打 0–10 分
3. 按 LLM 分数重排，取 top-3 给生成阶段

**为什么用 LLM-as-judge**：本机 bge-reranker / Cohere-rerank 都装不上（transformers 坏 + 没网）。LLM-as-judge 慢 + 贵，但**判别准、可解释**，正好用来学概念。生产里有 GPU 时换 bge-reranker-v2-m3 一行替换。

In [ ]:
RERANK_PROMPT = '''你是一个相关性评分器。给定用户问题和文档片段，输出 0 到 10 的整数分数。
10 = 文档直接回答了问题；5 = 部分相关；0 = 完全无关。
只输出一个数字，不要任何解释。

问题：{query}
文档：{doc}

分数：'''

def llm_rerank(query: str, candidates: list[int], top_k: int = 3) -> list[tuple[int, float]]:
    scored = []
    for doc_id in candidates:
        if MODE == 'ONLINE':
            resp = chat(RERANK_PROMPT.format(query=query, doc=DOCS[doc_id]))
            # 容错解析
            m = re.search(r'\d+', resp)
            score = float(m.group()) if m else 5.0
            score = max(0, min(10, score))
        else:
            # OFFLINE stub：用 token 重叠模拟「相关性打分」
            q_tokens = set(re.findall(r'[\u4e00-\u9fff]|[A-Za-z0-9]+', query.lower()))
            d_tokens = set(re.findall(r'[\u4e00-\u9fff]|[A-Za-z0-9]+', DOCS[doc_id].lower()))
            overlap = len(q_tokens & d_tokens)
            score = min(10.0, overlap * 2.0)
        scored.append((doc_id, score))
    scored.sort(key=lambda x: -x[1])
    return scored[:top_k]

In [ ]:
# Demo：对比「dense top3」与「dense top10 → rerank top3」
for q in ['XZ4054H 的升级版规格', 'PM 系列待机功耗', '什么是注意力机制']:
    dense_top10 = [d for d, _ in dense_search(q, top_k=10)]
    dense_top3  = dense_top10[:3]
    rerank_top3 = [d for d, _ in llm_rerank(q, dense_top10, top_k=3)]

    print(f'\nQ: {q!r}')
    print(f'  Dense top3              : {dense_top3}')
    print(f'  Dense top10 → Rerank top3: {rerank_top3}')
    # 看排序是否变化
    if dense_top3 != rerank_top3:
        print(f'  → 排序变了：rerank 把 {set(rerank_top3) - set(dense_top3)} 提到前 3')

## 2. HyDE（Hypothetical Document Embeddings）

**直觉**：query 通常很短（10 字）vs doc 通常较长（200 字）。短 query 的 embedding 容易飘。

**HyDE 做法**：让 LLM 先**幻想一段「如果这道题的答案是什么样子」**的假答案 → 用这段假答案的 embedding 去检索 → 更容易匹配真实答案的 doc。

**反直觉的事**：假答案是否正确根本不重要 —— 我们只需要它在 embedding 空间里**和真实答案落在同一区域**。

In [ ]:
HYDE_PROMPT = '''你是一个领域专家。请用 50 字以内写一段「假设的、可能是答案」的文字，回答下面问题。
不要说"我不知道"，即使不确定也要给出合理猜测。直接输出答案文字，不要前缀。

问题：{query}

假设的答案：'''

def hyde_search(query: str, top_k: int = 3) -> tuple[str, list[tuple[int, float]]]:
    if MODE == 'ONLINE':
        hypo = chat(HYDE_PROMPT.format(query=query)).strip()
    else:
        # OFFLINE stub：把 query 中关键词扩展
        hypo = f'关于{query}的描述：通常用于专业场景，规格典型值为...'
    # 用 hypo 的 embedding 去检索
    h = embed(hypo)
    sims = doc_vecs @ h
    hits = [(int(i), float(sims[i])) for i in np.argsort(-sims)[:top_k]]
    return hypo, hits

for q in ['XZ4054H 的容量', '什么是 RAG', 'PM 系列省电的型号']:
    hypo, hits = hyde_search(q)
    print(f'\nQ: {q!r}')
    print(f'  hypo: {hypo[:60]!r}')
    print(f'  hits: {[d for d, _ in hits]}')
    # 对比直接 dense
    direct = [d for d, _ in dense_search(q, top_k=3)]
    print(f'  vs direct dense: {direct}')

## 3. Multi-query —— 一题变 N 题

**思路**：让 LLM 把用户的 query 改写成 3-5 个**不同表述、但意思相同**的 query → 每个都检索 → 用 RRF 融合。

**收益**：covers 用户提问方式的多样性。「LoRA 微调」「low-rank adaptation」「参数高效微调」可能命中不同 doc。

In [ ]:
MULTI_QUERY_PROMPT = '''请把下面用户问题改写成 3 个不同表述（保留原意，换说法 / 加同义词 / 调语序）。
每行一个改写，不要编号、不要前缀。

原问题：{query}

改写：'''

def multi_query_rewrite(query: str, n: int = 3) -> list[str]:
    if MODE == 'ONLINE':
        resp = chat(MULTI_QUERY_PROMPT.format(query=query))
        rewrites = [line.strip() for line in resp.split('\n') if line.strip()][:n]
        return rewrites or [query]
    else:
        # OFFLINE stub：用同义词词典做简易改写
        synmap = {'什么': '是何', '解决': '处理', '问题': '困难', '型号': '产品编号', '规格': '参数'}
        out = [query]
        for k, v in synmap.items():
            if k in query and len(out) <= n:
                out.append(query.replace(k, v))
        return out[:n] if len(out) >= 2 else [query, query + ' 详情']

def rrf_fuse(rankings, k=60, top_k=5):
    score = {}
    for r in rankings:
        for rank, (d, _) in enumerate(r, start=1):
            score[d] = score.get(d, 0) + 1 / (k + rank)
    return sorted(score.items(), key=lambda x: -x[1])[:top_k]

def multi_query_search(query: str, top_k: int = 3):
    rewrites = multi_query_rewrite(query, n=3)
    rankings = [dense_search(q, top_k=10) for q in rewrites]
    fused = rrf_fuse(rankings, top_k=top_k)
    return rewrites, fused

for q in ['什么是 RAG 的核心问题', 'XZ4054H 升级型号', '低功耗芯片']:
    rewrites, hits = multi_query_search(q)
    print(f'\nQ: {q!r}')
    for r in rewrites:
        print(f'  rewrite: {r!r}')
    print(f'  fused top-3: {[d for d, _ in hits]}')

## 4. 三招组合 —— 生产 RAG 的标准 pipeline

```
user query
    │
    │  (可选) multi-query 改写 × 3
    ▼
[ dense + BM25 ] 各 top-20  ──RRF 融合──▶ candidate top-20
    │
    │  LLM-as-judge / cross-encoder rerank
    ▼
  top-3
    │
    │  LLM generate（拼 context + question）
    ▼
  answer
```

**HyDE 通常单独作为一条「替代检索路径」**，与 dense / BM25 并列后融合，而不是串行接在 query 后。

In [ ]:
def production_rag_recall(query: str, top_k_final: int = 3, recall_k: int = 10) -> list[int]:
    # 1. multi-query 改写
    rewrites = multi_query_rewrite(query, n=2)
    # 2. 每个 rewrite + 原 query 都跑 dense
    rankings = [dense_search(q, top_k=recall_k) for q in [query, *rewrites]]
    # 3. 加一路 HyDE
    _, hyde_hits = hyde_search(query, top_k=recall_k)
    rankings.append(hyde_hits)
    # 4. RRF 融合
    fused = rrf_fuse(rankings, top_k=recall_k)
    candidates = [d for d, _ in fused]
    # 5. LLM rerank
    reranked = llm_rerank(query, candidates, top_k=top_k_final)
    return [d for d, _ in reranked]

for q in ['XZ4054H 的升级版规格', 'RAG 解决幻觉问题', 'PM 待机省电']:
    direct = [d for d, _ in dense_search(q, top_k=3)]
    full   = production_rag_recall(q, top_k_final=3)
    print(f'\nQ: {q!r}')
    print(f'  直接 dense top3 : {direct}')
    print(f'  full pipeline   : {full}')
    diff = set(full) - set(direct)
    if diff:
        print(f'  → full pipeline 把 doc{list(diff)} 提到前 3')

## 5. 成本意识

每个高级技巧都不是白来的。**成本估算**（按生产 RAG 每次 query 算）：

| 阶段 | 调用次数 | 典型延迟 | 备注 |
|------|---------|---------|------|
| 朴素 dense | embed × 1 + 向量库 search × 1 | 30–100 ms | baseline |
| + BM25 hybrid | BM25 search × 1 + RRF | + 10–30 ms | BM25 极快 |
| + multi-query (3 rewrites) | LLM × 1 + embed × 3 + search × 3 | + 1–2 s | **LLM 调用最贵** |
| + HyDE | LLM × 1 + embed × 1 + search × 1 | + 0.5–1 s | LLM 短输出，不算贵 |
| + LLM rerank (10 候选) | LLM × 10 | + 5–10 s | **真贵**，生产用 cross-encoder rerank（10–50 ms） |

**实战经验**：
- query 多但每次成本敏感 → 跳过 multi-query，HyDE 看情况
- 召回质量优先 → 全开
- 用 LLM rerank 不可持续 → 换 bge-reranker（cross-encoder，~10-50 ms / pair on GPU）

## 深入思考

1. **为什么 reranker 比 dense 更准？**
   - dense 是「**双塔**」：query 和 doc 各自独立 embed，最后只比一个数（cosine）。reranker（cross-encoder）把 query 和 doc 拼起来送进 transformer，**让 attention 跨 query×doc 学相关性**，表达力强得多。代价是不能预 embed → 必须实时算。
2. **HyDE 不会让模型「自圆其说幻觉」吗？**
   - 不会。HyDE 只用 hypo 的 embedding，**真生成时还是只看实际 retrieved doc**。最多是 retrieved 错了 doc → 答案错。
3. **multi-query 的 N 选多少？**
   - 3-5 是甜点。再多 LLM 改写质量下降（开始重复或漂题），成本线性涨。
4. **三招都开了召回还烂怎么办？**
   - 多半是 ingest 阶段（chunk / loader）出了问题，回去重新设计。**检索阶段的优化有上限，数据阶段才是天花板**。
5. **生产里能不能把 LLM-as-judge 完全换掉？**
   - 能。**步骤**：(a) 用 LLM-as-judge 跑 1000 个 (query, doc) 对标注；(b) 用这些标注 fine-tune bge-reranker；(c) 上生产 reranker 又快又准。**数据飞轮**的一种。

**改一改**：
- 把 reranker 改成「只 rerank top-5」，看延迟是否大降、召回是否大降
- 在 ONLINE 模式跑，对比 stub 与真 LLM 的 rerank 结果

## 自检 ✅

- [ ] 解释 cross-encoder reranker 比双塔 dense 准的根本原因。
- [ ] 默背 HyDE 的 3 步流程（LLM 写假答案 → embed 假答案 → 检索真 doc）。
- [ ] 解释「multi-query 与 HyDE 各自解决什么问题」。
- [ ] 给一个延迟超标的 RAG，能列出 4 个降本路径。
- [ ] 解释「为什么 reranker 是 cross-encoder，不是另一个双塔模型」。

## 下一步

→ [`23_self_corrective_rag.ipynb`](23_self_corrective_rag.ipynb)